# 11 · CADD — a real, live deleteriousness score

Unlike the demo predictors, **CADD** is **REAL** here — fetched live from the CADD v1.7 REST API. CADD (Rentzsch et al. 2021, *Genome Medicine*, PMID 33618777) rolls dozens of annotations into one PHRED-scaled score. This notebook also teaches the single most important practical lesson: **validate genomic coordinates against the reference before trusting any tool.**

> ✅ **REAL / LIVE.** `tk.fetch_cadd(...)` calls the live CADD API. Requires network access. CADD PHRED **>= 15 ~ top 3%**, **>= 20 ~ top 1%**.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

In [2]:
# The CADD coordinate-validation demo below needs the curated splice variants.
splice = tk.load_splice_demo()

## 5 · CADD — a *real* deleteriousness score, fetched live

Everything above was demo. **CADD is real.**

**CADD** (Combined Annotation Dependent Depletion; the splice-aware v1.7 is
Rentzsch *et al.* 2021, *Genome Medicine*, PMID **33618777**) rolls dozens of
annotations — conservation, regulatory marks, and splice features — into **one**
integrated deleteriousness score. The convenient number is the **PHRED-scaled**
value:

- **PHRED ≥ 15** → roughly the **top 3%** most deleterious of all possible variants
- **PHRED ≥ 20** → roughly the **top 1%**

`tk.fetch_cadd(chrom, pos, ref, alt)` queries the **live CADD v1.7 REST API** and
returns `{'cadd_raw': ..., 'cadd_phred': ...}` (or `None` values on a miss).

> **Minus-strand gotcha.** CFTR sits on the genomic **minus strand**. So a change
> written on the *coding* strand (say `C>T`) appears on the genomic *plus* strand as
> its **complement** (`G>A`). CADD is indexed on the plus strand. `tk.fetch_cadd`
> tries **both orientations** for you, so you can pass coding-strand alleles and it
> still finds the match.

Let's make a **live** call on a position whose ref/alt genuinely matches the
reference genome.

> **Reproducibility.** Because CADD is queried *live*, **pin the version (v1.7, GRCh38)** and **cache the responses** so a re-run is stable and offline — a CADD version bump would change scores. Record the endpoint + version in `data_manifest.json`.

In [3]:
# LIVE call #1 — the position suggested in the toolkit smoke test.
# ref/alt here DO match the genome, so we get a real number back.
res1 = tk.fetch_cadd('7', 117548628, 'G', 'A')
print('7:117548628 G>A  ->', res1)
print(f"  CADD PHRED = {res1['cadd_phred']}  (low: near the benign end)")

7:117548628 G>A  -> {'cadd_raw': -0.964845, 'cadd_phred': 0.028}
  CADD PHRED = 0.028  (low: near the benign end)


That call **succeeded** and returned a real (low) PHRED. Now let's fetch one that
lands high — and let it exercise the minus-strand handling. The variant
`c.1210A>G` is listed with coding-strand alleles `A>G`; on the genomic plus strand
the reference base is actually `T`, and `tk.fetch_cadd` transparently complements
`A>G` → `T>C` to find the match.


In [4]:
# LIVE call #2 — a position that scores high, and that needs strand-complementing.
res2 = tk.fetch_cadd('7', 117548735, 'A', 'G')
print('7:117548735 A>G (coding strand)  ->', res2)
phred2 = res2['cadd_phred']
if phred2 is not None:
    band = 'top ~1% (>=20)' if phred2 >= 20 else ('top ~3% (>=15)' if phred2 >= 15 else 'below 15')
    print(f'  CADD PHRED = {phred2}  ->  {band}')
    print('  Note: coding A>G matched genomic T>C via strand complementing.')

7:117548735 A>G (coding strand)  -> {'cadd_raw': 4.353682, 'cadd_phred': 25.0}
  CADD PHRED = 25.0  ->  top ~1% (>=20)
  Note: coding A>G matched genomic T>C via strand complementing.


## 6 · 🧪 Data-quality lesson: validate coordinates against the reference *first*

This is the most important practical lesson in the notebook, so it gets its own
section.

The 9 curated splice variants have **hand-entered genomic coordinates**. Hand-entered
coordinates are exactly where silent errors creep in — a transposed digit, the wrong
strand, a base copied from the wrong build. And at least **one row here is wrong**.

Look at **`c.2988+1G>A`** (legacy `2988+1G>A`). The toolkit lists it as:

```
variant_id = 7-117592260-C-T     (chrom 7, pos 117592260, ref C, alt T)
```

But if you actually ask CADD what base sits at **7:117592260**, the reference genome
says the base there is **A** (the API offers alts `A>C`, `A>G`, `A>T`). So the
recorded **`ref='C'` does not match the reference genome**. The coordinate is simply
wrong.

**Consequence:** because `tk.fetch_cadd` only returns a score when the ref/alt matches
the reference (in either orientation), a wrong ref/alt means it finds **no match** and
returns `None`. The downstream tool fails **silently** — no error, no warning, just a
missing number that's easy to overlook.

Let's watch it happen live.


In [5]:
# The mismatched row, exactly as recorded in the toolkit.
bad = tk.fetch_cadd('7', 117592260, 'C', 'T')
print('as recorded  7:117592260 C>T  ->', bad)
print('  cadd_phred is', bad['cadd_phred'], '=> NO score. Silent failure.')
print()

# Ask the API what base actually lives at 7:117592260.
import requests
url = 'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/7:117592260-117592260'
rows = requests.get(url, timeout=15).json()
ref_bases = sorted({rec[2] for rec in rows[1:] if len(rec) >= 4})
print('reference base(s) the genome actually reports at 7:117592260:', ref_bases)
print("=> genome says 'A', but the toolkit row claims ref='C'. Coordinate is wrong.")

as recorded  7:117592260 C>T  -> {'cadd_raw': None, 'cadd_phred': None}
  cadd_phred is None => NO score. Silent failure.



reference base(s) the genome actually reports at 7:117592260: ['A']
=> genome says 'A', but the toolkit row claims ref='C'. Coordinate is wrong.


### The takeaway

> **Always validate `chrom:pos:ref:alt` against the reference genome *before* you
> trust it.** A single wrong `ref` base:
>
> - makes annotation tools (CADD, SpliceAI, VEP, …) return **nothing** for that
>   variant, usually **without any error**, and
> - if the wrong base *happens* to exist at a nearby position, you can get a
>   **plausible-looking but wrong** score — even more dangerous.
>
> **How to validate in practice:** confirm the `ref` allele matches the reference
> build (e.g. with `bcftools norm --check-ref`, Ensembl VEP, `pysam.FastaFile`, or —
> as we just did — by reading which alts an API offers at that position). Do it once,
> up front, for your whole variant list. It is the cheapest bug you'll ever prevent.

This is *why* several of the 9 demo variants return `None` from `tk.fetch_cadd`: their
hand-entered coordinates don't match GRCh38. Let's see the full pattern.


In [6]:
# Try a live CADD fetch for all 9 and mark which coordinates validate.
checks = []
for _, r in splice.iterrows():
    got = tk.fetch_cadd(r['chrom'], int(r['pos']), r['ref'], r['alt'])
    checks.append({
        'legacy_name': r['legacy_name'],
        'coord': f"{r['chrom']}:{r['pos']} {r['ref']}>{r['alt']}",
        'live_cadd_phred': got['cadd_phred'],
        'coord_validates': got['cadd_phred'] is not None,
    })
check_df = pd.DataFrame(checks)
print('coordinates that validate against GRCh38:',
      int(check_df['coord_validates'].sum()), 'of', len(check_df))
check_df

coordinates that validate against GRCh38: 4 of 9


,legacy_name,coord,live_cadd_phred,coord_validates
0,3849+10kb C>T,7:117645994 C>T,0.264,True
1,2789+5G>A,7:117587799 G>A,NaN,False
2,3272-26A>G,7:117616814 A>G,11.740,True
3,2657+3A>G,7:117587800 T>C,NaN,False
4,IVS8_5T,7:117548628 G>A,0.028,True
5,2988+1G>A,7:117592260 C>T,NaN,False
6,1811+1.6kbA>G,7:117559590 G>A,NaN,False
7,syn context,7:117548735 A>G,25.000,True
8,c.2657+120C>T,7:117588032 C>T,NaN,False


## Key takeaways

1. **CADD is REAL/live**: PHRED **>= 15 ~ top 3%**, **>= 20 ~ top 1%**. CFTR is on the **minus strand**, so `tk.fetch_cadd` complements coding-strand alleles to match the plus-strand reference.
2. **Validate `chrom:pos:ref:alt` first.** A wrong `ref` (like the mis-recorded `c.2988+1G>A`) makes annotation tools return `None` **silently**.
3. Several demo splice coordinates don't validate against GRCh38 — check before trusting.

**Next:** notebook 12 — **Integration**, where every tool is combined (and compared).